In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import re

# ── Input ──────────────────────────────────────────────────────────────────────
df = pd.read_csv('../data/cinestreamdata.csv')
df['release_cinema_wide'] = pd.to_datetime(df['release_cinema_wide'])

# Week of release → for total_playing and new_releases
df['bom_week'] = df['release_cinema_wide'].apply(
    lambda d: f"{d.isocalendar()[0]}W{d.isocalendar()[1]:02d}"
)

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/120.0.0.0 Safari/537.36'
}

THEATER_THRESHOLD = 600

# ── Helpers ────────────────────────────────────────────────────────────────────
def parse_int(value):
    cleaned = value.strip().replace(',', '')
    try:
        return int(cleaned)
    except ValueError:
        return None

def parse_table(bom_week):
    """Fetch and parse a BOM weekly page, return rows as list of dicts."""
    url = f"https://www.boxofficemojo.com/weekly/{bom_week}/"
    try:
        res = requests.get(url, headers=HEADERS, timeout=10)
        res.raise_for_status()
    except Exception as e:
        print(f"  ERROR fetching {url}: {e}")
        return None, None

    soup = BeautifulSoup(res.text, 'html.parser')
    table = soup.find('table')
    if not table:
        print(f"  No table found for {bom_week}")
        return None, None

    header_cells = table.find('tr').find_all('th')
    header_texts = [th.get_text(strip=True) for th in header_cells]

    try:
        weeks_idx    = header_texts.index('Weeks')
        theaters_idx = header_texts.index('Theaters')
    except ValueError as e:
        print(f"  Missing column for {bom_week}: {e}")
        return None, None

    tbody = table.find('tbody') or table
    rows = tbody.find_all('tr')
    if not table.find('tbody'):
        rows = rows[1:]

    parsed_rows = []
    for row in rows:
        cells = row.find_all('td')
        if not cells:
            continue
        parsed_rows.append({
            'theaters': parse_int(cells[theaters_idx].get_text(strip=True)) if theaters_idx < len(cells) else None,
            'weeks':    cells[weeks_idx].get_text(strip=True) if weeks_idx < len(cells) else None,
        })

    return parsed_rows, url

def get_competition(rows):
    """Compute total_playing and new_releases from parsed rows."""
    total_playing = 0
    new_releases  = 0
    for r in rows:
        if r['theaters'] is not None and r['theaters'] >= THEATER_THRESHOLD:
            total_playing += 1
            if r['weeks'] == '1':
                new_releases += 1
    return total_playing, new_releases

# ── Loop over every movie ──────────────────────────────────────────────────────
results = []
total = len(df)

# Cache pages to avoid re-fetching the same week twice
page_cache = {}

for i, row in df.iterrows():
    title    = row['title']
    bom_week = row['bom_week']

    print(f"[{i+1}/{total}] {title}")

    # Fetch competition week (cached)
    if bom_week not in page_cache:
        page_cache[bom_week], _ = parse_table(bom_week)
        time.sleep(0.2)
    comp_rows = page_cache[bom_week]

    # Competition counts from release week
    total_p, new_r = get_competition(comp_rows) if comp_rows else (None, None)

    results.append({
        'tconst':              row['tconst'],
        'title':               title,
        'release_cinema_wide': row['release_cinema_wide'].strftime('%Y-%m-%d'),
        'bom_week':            bom_week,
        'total_playing':       total_p,
        'new_releases':        new_r
    })

# ── Output ─────────────────────────────────────────────────────────────────────
out = pd.DataFrame(results)
out.to_csv('../data/part1_boxofficemojo.csv', index=False)
print(f"\nDone! {len(out)} rows saved to ../data/part1_boxofficemojo.csv")

C:\Users\stefv\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


[1/345] Motherless Brooklyn
[2/345] Alita Battle Angel
[3/345] Shazam!
[4/345] Silence
[5/345] Suburbicon
[6/345] Chips
[7/345] Rings
[8/345] The Last Full Measure
[9/345] Pet Sematary
[10/345] Super Troopers 2
[11/345] Jungle Cruise
[12/345] Fantasy Island
[13/345] The Little Things
[14/345] Unhinged
[15/345] Gemini Man
[16/345] Lightyear
[17/345] The Invisible Man
[18/345] Thor Love And Thunder
[19/345] The Killing Of Two Lovers
[20/345] Winchester
[21/345] Bill & Ted Face The Music
[22/345] Freaky
[23/345] Old
[24/345] Wrath Of Man
[25/345] Spirit Untamed
[26/345] House Of Gucci
[27/345] Licorice Pizza
[28/345] Together Together
[29/345] Rambo Last Blood
[30/345] Same Kind Of Different As Me
[31/345] Sonic The Hedgehog 2
[32/345] Holmes & Watson
[33/345] London Fields
[34/345] The Strangers Prey At Night
[35/345] Xxx Return Of Xander Cage
[36/345] The Hustle
[37/345] The Happytime Murders
[38/345] The Lost City
[39/345] Tomb Raider
[40/345] Downsizing
[41/345] It
[42/345] Blacklight